# Chapter 2. Organized Code Modules

This notebook is reorganized into numbered modules with short descriptions. Exploratory/debug cells have been removed or merged so the workflow is easier to follow.

## Module 1. Dataset Preview and Prompt Formatting

This module loads the CoS-E dataset and defines helper functions for turning multiple-choice options into the prompt format used later in training and evaluation.

In [2]:
from datasets import load_dataset

dataset = load_dataset("cos_e", "v1.11", split="train")
print(dataset)
print("columns:", dataset.column_names)


def choices_to_str(choices):
    # list/tuple: ["xx", "yy", ...] -> "(A) xx (B) yy ..."
    if isinstance(choices, (list, tuple)):
        return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(choices)])
    # dict: {"text":[...]} or similar
    if isinstance(choices, dict):
        if "text" in choices and isinstance(choices["text"], (list, tuple)):
            return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(choices["text"])])
        return str(choices)
    return str(choices)


ex = dataset[0]
text = (
    f"Question: {ex['question']}\n"
    f"Choices: {choices_to_str(ex['choices'])}\n"
    f"Answer: {ex['answer']}\n"
    f"Explanation: {ex['abstractive_explanation']}"
)

print(text)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

v1.11/train-00000-of-00001.parquet:   0%|          | 0.00/1.73M [00:00<?, ?B/s]

v1.11/validation-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'choices', 'answer', 'abstractive_explanation', 'extractive_explanation'],
    num_rows: 9741
})
columns: ['id', 'question', 'choices', 'answer', 'abstractive_explanation', 'extractive_explanation']
Question: "There are 10 apples on an apple tree.  Three fall off.  Now there are X apples."  What is this an example of?
Choices: (A) park (B) coloring book (C) garden center (D) math problem (E) gravity
Answer: math problem
Explanation: webmath is designed to help you solve


## Module 2. LoRA Training with Normal Prompt

This module contains the main training pipeline for GPT-2 + LoRA using the standard prompt format, including configuration, preprocessing, training, validation, and checkpoint saving.

In [ ]:
# =========================
# Install dependencies
# =========================
!pip -q install torch transformers datasets peft tqdm bert-score

import random
import re
import string
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from datasets import load_dataset

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    default_data_collator,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model

# =========================
# Reproducibility
# =========================
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Config (edit here)
# =========================
EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
MAX_LEN = 256
EXPL_KEY = "abstractive_explanation"
VAL_ACC_SAMPLES = 200
SAVE_DIR = "lora_gpt2_cose_ckpt"

# =========================
# Helper functions
# =========================
def choices_to_list(choices):
    if isinstance(choices, (list, tuple)):
        return list(choices)
    if isinstance(choices, dict) and "text" in choices and isinstance(choices["text"], (list, tuple)):
        return list(choices["text"])
    return [str(choices)]

def choices_to_str(choices):
    opts = choices_to_list(choices)
    return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(opts)])

def build_prompt(ex):
    return (
        f"Question: {ex['question']}\n"
        f"Choices: {choices_to_str(ex['choices'])}\n"
        f"Answer:"
    )

def build_target(ex):
    return f" {ex['answer']}\nExplanation: {ex[EXPL_KEY]}"

def normalize_text(s):
    s = str(s).strip().lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def map_pred_to_answer_text(pred, choices):
    pred_norm = normalize_text(pred)
    opts = choices_to_list(choices)

    m = re.match(r"^\(?([a-e])\)?$", pred_norm)
    if m:
        idx = ord(m.group(1)) - ord("a")
        if 0 <= idx < len(opts):
            return normalize_text(opts[idx])

    m = re.match(r"^\(?([a-e])\)?\s+(.*)$", pred_norm)
    if m:
        idx = ord(m.group(1)) - ord("a")
        if 0 <= idx < len(opts):
            return normalize_text(opts[idx])
        pred_norm = normalize_text(m.group(2))

    for opt in opts:
        opt_norm = normalize_text(opt)
        if pred_norm == opt_norm or pred_norm in opt_norm or opt_norm in pred_norm:
            return opt_norm

    return pred_norm

def parse_answer(generated_text):
    if "Answer:" not in generated_text:
        return ""
    after = generated_text.split("Answer:", 1)[1].strip()
    return after.split("\n", 1)[0].strip()

# =========================
# Load CoS-E dataset
# =========================
train_ds = load_dataset("cos_e", "v1.11", split="train")
val_ds = load_dataset("cos_e", "v1.11", split="validation")

print("Train columns:", train_ds.column_names)
print("Train size:", len(train_ds), "| Val size:", len(val_ds))

# =========================
# Tokenizer
# =========================
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


def tokenize_example(ex):
    prompt = build_prompt(ex)
    target = build_target(ex)
    full_text = prompt + target

    tok = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=False,
    )["input_ids"]
    prompt_len = min(len(prompt_ids), MAX_LEN)

    labels = tok["input_ids"].copy()
    for i in range(prompt_len):
        labels[i] = -100

    labels = [lab if am == 1 else -100 for lab, am in zip(labels, tok["attention_mask"])]
    tok["labels"] = labels
    return tok


train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_example, remove_columns=val_ds.column_names)

train_loader = DataLoader(
    train_tok,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=default_data_collator,
)
val_loader = DataLoader(
    val_tok,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=default_data_collator,
)

# =========================
# Validation helpers
# =========================
def compute_val_loss_for(model_to_eval):
    was_training = model_to_eval.training
    model_to_eval.eval()

    total, n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            out = model_to_eval(input_ids=input_ids, attention_mask=attn, labels=labels)
            total += float(out.loss.detach().cpu().item())
            n += 1

    if was_training:
        model_to_eval.train()
    return total / max(n, 1)


def eval_val_accuracy_for(model_to_eval, n_samples=200):
    was_training = model_to_eval.training
    model_to_eval.eval()

    correct = 0
    m = min(n_samples, len(val_ds))

    with torch.no_grad():
        for i in range(m):
            ex = val_ds[i]
            prompt = build_prompt(ex)
            inputs = tokenizer(prompt, return_tensors="pt").to(device)

            gen_ids = model_to_eval.generate(
                **inputs,
                max_new_tokens=24,
                do_sample=False,
            )
            decoded = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

            pred = parse_answer(decoded)
            pred_norm = map_pred_to_answer_text(pred, ex["choices"])
            gold_norm = normalize_text(ex["answer"])

            if pred_norm == gold_norm:
                correct += 1

    if was_training:
        model_to_eval.train()
    return correct / m

# =========================
# Baseline: GPT-2 before LoRA
# =========================
base_eval_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
base_val_loss = compute_val_loss_for(base_eval_model)
base_val_acc = eval_val_accuracy_for(base_eval_model, n_samples=VAL_ACC_SAMPLES)
print(f"[Baseline GPT-2] val_loss={base_val_loss:.4f} | val_acc@{VAL_ACC_SAMPLES}={base_val_acc:.3f}")

del base_eval_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================
# Model: GPT-2 + LoRA
# =========================
base_model = GPT2LMHeadModel.from_pretrained("gpt2")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config).to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# =========================
# Training loop
# =========================
best_val_loss = float("inf")
optimizer.zero_grad()

for ep in range(1, EPOCHS + 1):
    pbar = tqdm(train_loader, desc=f"Epoch {ep}/{EPOCHS}")

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(input_ids=input_ids, attention_mask=attn, labels=labels)
        loss = out.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        pbar.set_postfix(train_loss=float(loss.detach().cpu().item()))

    val_loss = compute_val_loss_for(model)
    val_acc = eval_val_accuracy_for(model, n_samples=VAL_ACC_SAMPLES)
    print(f"[Epoch {ep}] val_loss={val_loss:.4f} | val_acc@{VAL_ACC_SAMPLES}={val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"Best checkpoint updated -> {SAVE_DIR} (val_loss={best_val_loss:.4f})")

print("Training done. Best checkpoint:", SAVE_DIR)



Device: cuda
Train columns: ['id', 'question', 'choices', 'answer', 'abstractive_explanation', 'extractive_explanation']
Train size: 9741 | Val size: 1221


Map:   0%|          | 0/9741 [00:00<?, ? examples/s]

Map:   0%|          | 0/1221 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

...

[Epoch 1] val_loss=2.1227 | val_acc@200=0.335
Best checkpoint updated -> lora_gpt2_cose_ckpt (val_loss=2.1227)


Epoch 2/3: 100%|██████████| 2436/2436 [02:16<00:00, 17.85it/s, train_loss=0.824]
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open

[Epoch 2] val_loss=2.0840 | val_acc@200=0.330
Best checkpoint updated -> lora_gpt2_cose_ckpt (val_loss=2.0840)


Epoch 3/3: 100%|██████████| 2436/2436 [02:16<00:00, 17.85it/s, train_loss=2.74]
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-

[Epoch 3] val_loss=2.0817 | val_acc@200=0.340
Best checkpoint updated -> lora_gpt2_cose_ckpt (val_loss=2.0817)
Training done. Best checkpoint: lora_gpt2_cose_ckpt


## Module 2B. Baseline Model (Full Fine-Tuning GPT-2)

This module adds a stronger baseline than zero-shot GPT-2 by fine-tuning the full GPT-2 model on the same normal-prompt CoS-E task. It uses the same preprocessing and evaluation style as the LoRA experiment so the comparison is fairer.

In [3]:
# =========================
# Baseline model: full fine-tuning GPT-2 on normal prompt
# =========================
!pip -q install torch transformers datasets tqdm pandas

import os
import re
import gc
import string
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    default_data_collator,
    get_linear_schedule_with_warmup,
)

# -------------------------
# Reproducibility
# -------------------------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -------------------------
# Config
# -------------------------
MODEL_NAME = "gpt2"
BASELINE_EPOCHS = 3
BASELINE_BATCH_SIZE = 4
BASELINE_LR = 5e-5
BASELINE_WEIGHT_DECAY = 0.01
BASELINE_WARMUP_RATIO = 0.06
BASELINE_GRAD_CLIP = 1.0
BASELINE_MAX_LEN = 256
BASELINE_VAL_ACC_SAMPLES = 200
BASELINE_SAVE_DIR = "./baseline_gpt2_cose_ckpt"
BASELINE_RESULTS_CSV = os.path.join(BASELINE_SAVE_DIR, "baseline_results.csv")

os.makedirs(BASELINE_SAVE_DIR, exist_ok=True)

# -------------------------
# Helpers
# -------------------------
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def choices_to_list(choices):
    if isinstance(choices, (list, tuple)):
        return list(choices)
    if isinstance(choices, dict) and "text" in choices and isinstance(choices["text"], (list, tuple)):
        return list(choices["text"])
    return [str(choices)]

def choices_to_str(choices):
    opts = choices_to_list(choices)
    return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(opts)])

def build_prompt(ex):
    return (
        f"Question: {ex['question']}\n"
        f"Choices: {choices_to_str(ex['choices'])}\n"
        f"Answer:"
    )

def build_target(ex):
    return f" {ex['answer']}\nExplanation: {ex['abstractive_explanation']}"

def normalize_text(s):
    s = str(s).strip().lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def map_pred_to_answer_text(pred, choices):
    pred_norm = normalize_text(pred)
    opts = choices_to_list(choices)

    m = re.match(r"^\(?([a-e])\)?$", pred_norm)
    if m:
        idx = ord(m.group(1)) - ord("a")
        if 0 <= idx < len(opts):
            return normalize_text(opts[idx])

    m = re.match(r"^\(?([a-e])\)?\s+(.*)$", pred_norm)
    if m:
        idx = ord(m.group(1)) - ord("a")
        if 0 <= idx < len(opts):
            return normalize_text(opts[idx])
        pred_norm = normalize_text(m.group(2))

    for opt in opts:
        opt_norm = normalize_text(opt)
        if pred_norm == opt_norm or pred_norm in opt_norm or opt_norm in pred_norm:
            return opt_norm

    return pred_norm

def parse_answer(generated_text):
    if "Answer:" not in generated_text:
        return ""
    after = generated_text.split("Answer:", 1)[1].strip()
    if "Explanation:" in after:
        after = after.split("Explanation:", 1)[0].strip()
    return after.split("\n", 1)[0].strip()

# -------------------------
# Data
# -------------------------
train_ds = load_dataset("cos_e", "v1.11", split="train")
val_ds = load_dataset("cos_e", "v1.11", split="validation")

print("Train size:", len(train_ds), "| Val size:", len(val_ds))

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_example(ex):
    prompt = build_prompt(ex)
    target = build_target(ex)
    full_text = prompt + target

    tok = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=BASELINE_MAX_LEN,
    )

    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=BASELINE_MAX_LEN,
        add_special_tokens=False,
    )["input_ids"]

    prompt_len = min(len(prompt_ids), BASELINE_MAX_LEN)
    labels = tok["input_ids"].copy()

    for i in range(prompt_len):
        labels[i] = -100

    labels = [lab if am == 1 else -100 for lab, am in zip(labels, tok["attention_mask"])]
    tok["labels"] = labels
    return tok

train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_example, remove_columns=val_ds.column_names)

train_loader = DataLoader(
    train_tok,
    batch_size=BASELINE_BATCH_SIZE,
    shuffle=True,
    collate_fn=default_data_collator,
)

val_loader = DataLoader(
    val_tok,
    batch_size=BASELINE_BATCH_SIZE,
    shuffle=False,
    collate_fn=default_data_collator,
)

# -------------------------
# Model
# -------------------------
baseline_ft_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
baseline_ft_model.config.pad_token_id = tokenizer.eos_token_id
baseline_ft_model.to(device)

optimizer = torch.optim.AdamW(
    baseline_ft_model.parameters(),
    lr=BASELINE_LR,
    weight_decay=BASELINE_WEIGHT_DECAY,
)

num_train_steps = BASELINE_EPOCHS * len(train_loader)
num_warmup_steps = int(BASELINE_WARMUP_RATIO * num_train_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_train_steps,
)

def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0.0
    total_steps = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()
            total_steps += 1
    return total_loss / max(total_steps, 1)

def evaluate_accuracy(model, dataset, n_samples=200):
    model.eval()
    total = min(n_samples, len(dataset))
    correct = 0

    with torch.no_grad():
        for i in tqdm(range(total), desc="baseline val acc"):
            ex = dataset[i]
            prompt = build_prompt(ex)
            inputs = tokenizer(prompt, return_tensors="pt").to(device)

            gen_ids = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            decoded = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
            pred = parse_answer(decoded)
            pred_norm = map_pred_to_answer_text(pred, ex["choices"])
            gold_norm = normalize_text(ex["answer"])

            if pred_norm == gold_norm:
                correct += 1

    return correct / total

history = []

for epoch in range(BASELINE_EPOCHS):
    baseline_ft_model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Baseline epoch {epoch+1}/{BASELINE_EPOCHS}")
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = baseline_ft_model(**batch)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(baseline_ft_model.parameters(), BASELINE_GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / max(len(train_loader), 1)
    val_loss = evaluate_loss(baseline_ft_model, val_loader)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
    })

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

baseline_val_acc = evaluate_accuracy(
    baseline_ft_model,
    val_ds,
    n_samples=BASELINE_VAL_ACC_SAMPLES,
)

print(f"Baseline fine-tuned GPT-2 validation accuracy@{BASELINE_VAL_ACC_SAMPLES}: {baseline_val_acc:.4f}")

baseline_ft_model.save_pretrained(BASELINE_SAVE_DIR)
tokenizer.save_pretrained(BASELINE_SAVE_DIR)

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(BASELINE_SAVE_DIR, "baseline_training_history.csv"), index=False)

results_df = pd.DataFrame([{
    "model": "GPT-2 full fine-tuning",
    "prompt": "Normal",
    "epochs": BASELINE_EPOCHS,
    "batch_size": BASELINE_BATCH_SIZE,
    "learning_rate": BASELINE_LR,
    "val_accuracy_at_200": baseline_val_acc,
}])
results_df.to_csv(BASELINE_RESULTS_CSV, index=False)

print(f"Saved baseline model to: {BASELINE_SAVE_DIR}")
print(f"Saved baseline metrics to: {BASELINE_RESULTS_CSV}")

cleanup()

Device: cuda
Train size: 9741 | Val size: 1221


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 3: train_loss=1.1794 | val_loss=2.2585


baseline val acc: 100%|██████████| 200/200 [01:29<00:00,  2.24it/s]

Baseline fine-tuned GPT-2 validation accuracy@200: 0.3050


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved baseline model to: ./baseline_gpt2_cose_ckpt
Saved baseline metrics to: ./baseline_gpt2_cose_ckpt/baseline_results.csv


## Module 3. Inference Demo

This module loads a saved LoRA checkpoint and runs generation on one validation example so you can inspect the predicted answer and explanation qualitatively.

In [4]:
import os
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from peft import PeftModel
from datasets import load_dataset


def choices_to_list(choices):
    if isinstance(choices, (list, tuple)):
        return list(choices)
    if isinstance(choices, dict) and "text" in choices and isinstance(choices["text"], (list, tuple)):
        return list(choices["text"])
    return [str(choices)]


def choices_to_str(choices):
    opts = choices_to_list(choices)
    return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(opts)])


device = "cuda" if torch.cuda.is_available() else "cpu"
model_dir = "/content/lora_gpt2_cose_ckpt"

if not os.path.exists(model_dir):
    raise FileNotFoundError(
        f"LoRA checkpoint folder not found: {model_dir}. "
        "Please upload and unzip lora_gpt2_cose_ckpt.zip first."
    )

required_files = ["adapter_config.json", "adapter_model.safetensors"]
missing_files = [f for f in required_files if not os.path.exists(os.path.join(model_dir, f))]
if missing_files:
    raise FileNotFoundError(
        "Missing required LoRA checkpoint files: " + ", ".join(missing_files)
    )

# Load tokenizer from checkpoint folder
tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base GPT-2 + LoRA adapter
base = GPT2LMHeadModel.from_pretrained("gpt2")
base.config.pad_token_id = tokenizer.eos_token_id
base = base.to(device)

model = PeftModel.from_pretrained(base, model_dir).to(device)
model.eval()

# Load validation data
val_ds = load_dataset("cos_e", "v1.11", split="validation")

# Pick an example by index
sample_idx = 3
ex = val_ds[sample_idx]

prompt = (
    f"Question: {ex['question']}\n"
    f"Choices: {choices_to_str(ex['choices'])}\n"
    f"Answer:"
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    gen_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

output = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

print("device:", device)
print("model_dir:", model_dir)
print("checkpoint files:", sorted(os.listdir(model_dir)))
print("\n=== Prompt ===")
print(prompt)
print("\n=== Model Output ===")
print(output)
print("\n=== Ground Truth ===")
print("Answer:", ex["answer"])
print("Explanation:", ex["abstractive_explanation"])


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


device: cuda
model_dir: /content/lora_gpt2_cose_ckpt
checkpoint files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json']

=== Prompt ===
Question: A farmer sees a weasel in the woods, where is the farmer?
Choices: (A) chicken coop (B) beach (C) fairytale (D) great outdoors (E) corn fields
Answer:

=== Model Output ===
Question: A farmer sees a weasel in the woods, where is the farmer?
Choices: (A) chicken coop (B) beach (C) fairytale (D) great outdoors (E) corn fields
Answer: fairytale
Explanation: fairytale farmer sees a weasel in the woods, where is the farmer fairytale

=== Ground Truth ===
Answer: great outdoors
Explanation: woods are must


## Module 4. Explanation Quality Evaluation (BERTScore)

This module compares explanation quality between baseline GPT-2 and LoRA-GPT-2 on the validation set using BERTScore.

In [ ]:
# =========================
# BERTScore: baseline GPT-2 vs LoRA on validation explanations
# =========================
import torch
from bert_score import score as bert_score
from transformers import GPT2LMHeadModel
from transformers.utils import logging as hf_logging
import warnings
import os

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def parse_explanation(generated_text):
    if "Explanation:" in generated_text:
        return generated_text.split("Explanation:", 1)[1].strip()

    if "Answer:" in generated_text:
        after = generated_text.split("Answer:", 1)[1]
        parts = after.split("\n", 1)
        if len(parts) > 1:
            return parts[1].strip()

    return ""


def compute_bertscore_for_model(model_to_eval, n_samples=200, desc="BERTScore eval"):
    was_training = model_to_eval.training
    model_to_eval.eval()

    m = min(n_samples, len(val_ds))
    cands, refs = [], []
    skipped = 0

    with torch.no_grad():
        for i in range(m):
            ex = val_ds[i]
            prompt = (
                f"Question: {ex['question']}\n"
                f"Choices: {choices_to_str(ex['choices'])}\n"
                f"Answer:"
            )

            inputs = tokenizer(prompt, return_tensors="pt").to(device)

            gen_ids = model_to_eval.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

            output = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
            pred_expl = parse_explanation(output)
            gold_expl = str(ex['abstractive_explanation']).strip()

            if pred_expl and gold_expl:
                cands.append(pred_expl)
                refs.append(gold_expl)
            else:
                skipped += 1

    if was_training:
        model_to_eval.train()

    if not cands:
        raise ValueError(f"{desc}: no valid prediction/reference explanation pairs found.")

    P, R, F1 = bert_score(
        cands,
        refs,
        lang='en',
        rescale_with_baseline=False,
        device=device,
        verbose=False,
    )

    return {
        "used": len(cands),
        "skipped": skipped,
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item(),
    }


BERT_SCORE_SAMPLES = 200

# Baseline GPT-2
baseline_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
baseline_metrics = compute_bertscore_for_model(
    baseline_model,
    n_samples=BERT_SCORE_SAMPLES,
)

del baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# LoRA model (expects `model` from inference cell)
lora_metrics = compute_bertscore_for_model(
    model,
    n_samples=BERT_SCORE_SAMPLES,
)

print("[Baseline GPT-2]")
print(f"samples used: {baseline_metrics['used']} (skipped: {baseline_metrics['skipped']})")
print(f"P: {baseline_metrics['precision']:.4f} | R: {baseline_metrics['recall']:.4f} | F1: {baseline_metrics['f1']:.4f}")

print("\n[LoRA GPT-2]")
print(f"samples used: {lora_metrics['used']} (skipped: {lora_metrics['skipped']})")
print(f"P: {lora_metrics['precision']:.4f} | R: {lora_metrics['recall']:.4f} | F1: {lora_metrics['f1']:.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[Baseline GPT-2]
samples used: 162 (skipped: 38)
P: 0.7838 | R: 0.8484 | F1: 0.8146

[LoRA GPT-2]
samples used: 200 (skipped: 0)
P: 0.7771 | R: 0.8397 | F1: 0.8070


**Qualitative note.** In a representative example, the model produced a fluent explanation but still selected the wrong answer. This suggests explanation fluency does not necessarily guarantee decision faithfulness.

## Module 5. LoRA Rank Ablation

This module studies how different LoRA ranks affect validation accuracy. It trains/evaluates multiple rank settings, saves the results, and summarizes the outcome in a clean table.

In [ ]:
# ============================================================
# Baseline GPT-2 + LoRA rank ablation
# Save results to CSV and display a clean table
# ============================================================

!pip -q install torch transformers datasets peft tqdm bert-score pandas

import os
import gc
import re
import string
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from datasets import load_dataset
from IPython.display import display

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    default_data_collator,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model

# =========================
# Reproducibility
# =========================
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Config
# =========================
MODEL_NAME = "gpt2"
EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
MAX_LEN = 256
VAL_ACC_SAMPLES = 200
SAVE_ROOT = "./lora_rank_ablation_outputs"
CSV_PATH = os.path.join(SAVE_ROOT, "rank_ablation_results.csv")
LORA_RANKS = [4, 8, 16]

os.makedirs(SAVE_ROOT, exist_ok=True)

# =========================
# Dataset
# =========================
train_ds = load_dataset("cos_e", "v1.11", split="train")
val_ds = load_dataset("cos_e", "v1.11", split="validation")

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

# =========================
# Tokenizer
# =========================
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# =========================
# Helpers
# =========================
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def choices_to_list(choices):
    if isinstance(choices, list):
        return choices
    if isinstance(choices, dict):
        if "text" in choices:
            return choices["text"]
        if "label" in choices and "text" in choices:
            return choices["text"]
    return list(choices)

def choices_to_str(choices):
    opts = choices_to_list(choices)
    return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(opts)])

def build_prompt(ex):
    return (
        f"Question: {ex['question']}\n"
        f"Choices: {choices_to_str(ex['choices'])}\n"
        f"Answer:"
    )

def build_target(ex):
    return f" {ex['answer']}\nExplanation: {ex['abstractive_explanation']}"

def tokenize_example(ex):
    prompt = build_prompt(ex)
    target = build_target(ex)
    full_text = prompt + target

    tok = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=False
    )["input_ids"]

    prompt_len = min(len(prompt_ids), MAX_LEN)
    labels = tok["input_ids"].copy()

    # mask prompt tokens
    for i in range(prompt_len):
        labels[i] = -100

    # mask padding tokens
    labels = [lab if attn == 1 else -100 for lab, attn in zip(labels, tok["attention_mask"])]
    tok["labels"] = labels
    return tok

def normalize_text(s):
    s = str(s).strip().lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def parse_answer(generated_text):
    if "Answer:" not in generated_text:
        return ""
    after = generated_text.split("Answer:", 1)[1].strip()
    ans_line = after.split("\n", 1)[0].strip()
    return ans_line

def answer_to_index(answer):
    # gold answer in CoS-E is often like "A", "B", ...
    answer = str(answer).strip().upper()
    if len(answer) == 1 and answer in "ABCDE":
        return ord(answer) - ord("A")
    return None

def map_prediction_to_index(pred_text, choices):
    pred = pred_text.strip()
    opts = choices_to_list(choices)

    # Case 1: exact letter
    m = re.match(r"^\(?([A-Ea-e])\)?$", pred)
    if m:
        return ord(m.group(1).upper()) - ord("A")

    # Case 2: starts with letter
    m = re.match(r"^\(?([A-Ea-e])\)?[\s:\-\,\.].*$", pred)
    if m:
        return ord(m.group(1).upper()) - ord("A")

    pred_norm = normalize_text(pred)

    # Case 3: match option text
    for i, opt in enumerate(opts):
        opt_norm = normalize_text(opt)
        if pred_norm == opt_norm or pred_norm in opt_norm or opt_norm in pred_norm:
            return i

    return None

# =========================
# Tokenize once
# =========================
train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_example, remove_columns=val_ds.column_names)

train_loader = DataLoader(
    train_tok,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=default_data_collator,
)

val_loader = DataLoader(
    val_tok,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=default_data_collator,
)

# =========================
# Evaluation
# =========================
def compute_val_loss(model):
    model.eval()
    total_loss = 0.0
    steps = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
            steps += 1

    return total_loss / max(steps, 1)

def compute_val_acc(model, dataset, num_samples=200):
    model.eval()
    correct = 0
    total = min(num_samples, len(dataset))

    for i in tqdm(range(total), desc="val_acc"):
        ex = dataset[i]
        prompt = build_prompt(ex)

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        pred_text = parse_answer(generated)

        pred_idx = map_prediction_to_index(pred_text, ex["choices"])
        gold_idx = answer_to_index(ex["answer"])

        if pred_idx is not None and gold_idx is not None and pred_idx == gold_idx:
            correct += 1

    return correct / total if total > 0 else 0.0

# =========================
# Train function
# =========================
def train_model(model, run_name):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(WARMUP_RATIO * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_val_loss = float("inf")
    best_dir = os.path.join(SAVE_ROOT, run_name)
    os.makedirs(best_dir, exist_ok=True)

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        loop = tqdm(train_loader, desc=f"{run_name} | Epoch {epoch+1}/{EPOCHS}")
        for batch in loop:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            loop.set_postfix(train_loss=loss.item())

        avg_train_loss = running_loss / max(len(train_loader), 1)
        val_loss = compute_val_loss(model)
        val_acc = compute_val_acc(model, val_ds, num_samples=VAL_ACC_SAMPLES)

        print(f"[{run_name}] Epoch {epoch+1}: train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | val_acc@{VAL_ACC_SAMPLES}={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model.save_pretrained(best_dir)
            tokenizer.save_pretrained(best_dir)

    return best_dir

# =========================
# Run baseline
# =========================
results = []

cleanup()
baseline_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
baseline_model.config.pad_token_id = tokenizer.pad_token_id
baseline_model = baseline_model.to(device)

baseline_val_loss = compute_val_loss(baseline_model)
baseline_val_acc = compute_val_acc(baseline_model, val_ds, num_samples=VAL_ACC_SAMPLES)

results.append({
    "model": "baseline_gpt2",
    "lora_rank": 0,
    "val_loss": round(baseline_val_loss, 4),
    f"val_acc@{VAL_ACC_SAMPLES}": round(baseline_val_acc, 4),
    "checkpoint_dir": "N/A"
})

print("\nBaseline done.\n")

# =========================
# Run LoRA ranks
# =========================
for r in LORA_RANKS:
    cleanup()
    run_name = f"gpt2_lora_r{r}"

    base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
    base_model.config.pad_token_id = tokenizer.pad_token_id

    lora_config = LoraConfig(
        r=r,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"],
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    best_dir = train_model(model, run_name)

    # reload best checkpoint for clean evaluation
    cleanup()
    base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
    base_model.config.pad_token_id = tokenizer.pad_token_id
    model = get_peft_model(base_model, lora_config)
    model.load_adapter(best_dir, adapter_name="default")
    model = model.to(device)

    val_loss = compute_val_loss(model)
    val_acc = compute_val_acc(model, val_ds, num_samples=VAL_ACC_SAMPLES)

    results.append({
        "model": "gpt2_lora",
        "lora_rank": r,
        "val_loss": round(val_loss, 4),
        f"val_acc@{VAL_ACC_SAMPLES}": round(val_acc, 4),
        "checkpoint_dir": best_dir
    })

# =========================
# Save + display results
# =========================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=[f"val_acc@{VAL_ACC_SAMPLES}", "val_loss"], ascending=[False, True]).reset_index(drop=True)

results_df.to_csv(CSV_PATH, index=False)

print("\n========== Final Results ==========")
display(results_df)

print(f"\nCSV saved to: {CSV_PATH}")

Device: cuda
Train size: 9741
Val size: 1221


Map:   0%|          | 0/9741 [00:00<?, ? examples/s]

Map:   0%|          | 0/1221 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

val_acc: 100%|██████████| 200/200 [01:32<00:00,  2.15it/s]


...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

val_acc: 100%|██████████| 200/200 [02:22<00:00,  1.40it/s]


========== Final Results ==========


,model,lora_rank,val_loss,val_acc@200,checkpoint_dir
0,gpt2_lora,8,2.0808,0.0,./lora_rank_ablation_outputs/gpt2_lora_r8
1,gpt2_lora,16,2.0875,0.0,./lora_rank_ablation_outputs/gpt2_lora_r16
2,gpt2_lora,4,2.0921,0.0,./lora_rank_ablation_outputs/gpt2_lora_r4
3,baseline_gpt2,0,3.9404,0.0,N/A



CSV saved to: ./lora_rank_ablation_outputs/rank_ablation_results.csv


In [ ]:
import os
import re
import string
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "gpt2"
VAL_ACC_SAMPLES = 200
SAVE_ROOT = "./lora_rank_ablation_outputs"
OUT_CSV = os.path.join(SAVE_ROOT, "rank_ablation_results_fixed_acc.csv")

# -------------------------
# Load validation set
# -------------------------
val_ds = load_dataset("cos_e", "v1.11", split="validation")

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# -------------------------
# Helpers
# -------------------------
def choices_to_list(choices):
    if isinstance(choices, list):
        return choices
    if isinstance(choices, dict) and "text" in choices:
        return choices["text"]
    return list(choices)

def choices_to_str(choices):
    opts = choices_to_list(choices)
    return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(opts)])

def build_prompt(ex):
    return (
        f"Question: {ex['question']}\n"
        f"Choices: {choices_to_str(ex['choices'])}\n"
        f"Answer:"
    )

def parse_answer(generated_text):
    if "Answer:" not in generated_text:
        return ""

    after = generated_text.split("Answer:", 1)[1].strip()

    # Stop at Explanation if it appears on same line
    if "Explanation:" in after:
        after = after.split("Explanation:", 1)[0].strip()

    # Keep first line only
    line = after.split("\n")[0].strip()

    # Extract first option letter if present
    m = re.search(r"\b([A-Ea-e])\b", line)
    if m:
        return m.group(1).upper()

    return line.strip()

def answer_to_index(answer, choices):
    opts = choices_to_list(choices)
    ans_norm = normalize_text(answer)

    for i, opt in enumerate(opts):
        opt_norm = normalize_text(opt)
        if ans_norm == opt_norm:
            return i

    # fallback: containment match
    for i, opt in enumerate(opts):
        opt_norm = normalize_text(opt)
        if ans_norm in opt_norm or opt_norm in ans_norm:
            return i

    return None

def normalize_text(s):
    s = str(s).strip().lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def map_prediction_to_index(pred_text, choices):
    pred = str(pred_text).strip().upper()

    # Case 1: direct option letter
    if pred in ["A", "B", "C", "D", "E"]:
        return ord(pred) - ord("A")

    # Case 2: something like "(A)"
    m = re.match(r"^\(?([A-E])\)?$", pred)
    if m:
        return ord(m.group(1)) - ord("A")

    # Case 3: text match against options
    pred_norm = normalize_text(pred_text)
    opts = choices_to_list(choices)
    for i, opt in enumerate(opts):
        opt_norm = normalize_text(opt)
        if pred_norm == opt_norm or pred_norm in opt_norm or opt_norm in pred_norm:
            return i

    return None

def compute_val_acc(model, dataset, num_samples=200, debug_first_n=5):
    model.eval()
    correct = 0
    total = min(num_samples, len(dataset))

    for i in tqdm(range(total), desc="re-eval acc"):
        ex = dataset[i]
        prompt = build_prompt(ex)

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        pred_text = parse_answer(generated)

        pred_idx = map_prediction_to_index(pred_text, ex["choices"])
        gold_idx = answer_to_index(ex["answer"], ex["choices"])

        if i < debug_first_n:
            print("PROMPT:", prompt)
            print("GENERATED:", generated)
            print("PARSED:", pred_text)
            print("PRED IDX:", pred_idx, "GOLD IDX:", gold_idx)
            print("GOLD ANSWER:", ex["answer"])
            print("-" * 80)

        if pred_idx is not None and gold_idx is not None and pred_idx == gold_idx:
            correct += 1

    return correct / total if total > 0 else 0.0

# -------------------------
# Re-evaluate baseline
# -------------------------
results = []

baseline_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
baseline_model.config.pad_token_id = tokenizer.pad_token_id
baseline_model = baseline_model.to(device)

baseline_acc = compute_val_acc(baseline_model, val_ds, num_samples=VAL_ACC_SAMPLES)

results.append({
    "model": "baseline_gpt2",
    "lora_rank": 0,
    f"val_acc@{VAL_ACC_SAMPLES}": round(baseline_acc, 4),
    "checkpoint_dir": "N/A"
})

# -------------------------
# Re-evaluate saved LoRA checkpoints
# -------------------------
for r in [4, 8, 16]:
    ckpt_dir = os.path.join(SAVE_ROOT, f"gpt2_lora_r{r}")
    print(f"\nLoading checkpoint: {ckpt_dir}")

    base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
    base_model.config.pad_token_id = tokenizer.pad_token_id

    model = PeftModel.from_pretrained(base_model, ckpt_dir)
    model = model.to(device)

    acc = compute_val_acc(model, val_ds, num_samples=VAL_ACC_SAMPLES)

    results.append({
        "model": "gpt2_lora",
        "lora_rank": r,
        f"val_acc@{VAL_ACC_SAMPLES}": round(acc, 4),
        "checkpoint_dir": ckpt_dir
    })

results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(OUT_CSV, index=False)
print(f"\nSaved fixed accuracy csv to: {OUT_CSV}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

re-eval acc:   0%|          | 1/200 [00:00<01:34,  2.10it/s]

PROMPT: Question: A beaver is know for building prowess, their supplies come from where?
Choices: (A) british columbia (B) body of water (C) wooded area (D) pay debts (E) zoo
Answer:
GENERATED: Question: A beaver is know for building prowess, their supplies come from where?
Choices: (A) british columbia (B) body of water (C) wooded area (D) pay debts (E) zoo
Answer: A beaver is known for building prowess, their supplies come from where?
Choices: (A) british columbia (B) body of water (C) wooded area
PARSED: A
PRED IDX: 0 GOLD IDX: 2
GOLD ANSWER: wooded area
--------------------------------------------------------------------------------


re-eval acc:   1%|          | 2/200 [00:00<01:34,  2.10it/s]

PROMPT: Question: A car was hailed to chauffeur someone to the opera house, where was it heading?
Choices: (A) go downtown (B) appear suddenly (C) go fast (D) bottom out (E) east
Answer:
GENERATED: Question: A car was hailed to chauffeur someone to the opera house, where was it heading?
Choices: (A) go downtown (B) appear suddenly (C) go fast (D) bottom out (E) east
Answer: (A) go downtown (B) appear suddenly (C) go fast (D) bottom out (E) east
Answer: (A) go downtown (B) appear suddenly (C)
PARSED: A
PRED IDX: 0 GOLD IDX: 0
GOLD ANSWER: go downtown
--------------------------------------------------------------------------------


...

re-eval acc:   2%|▎         | 5/200 [00:03<02:21,  1.38it/s]

PROMPT: Question: A gentleman is carrying equipment for golf, what is he likely to have?
Choices: (A) club (B) assembly hall (C) meditation center (D) meeting (E) church
Answer:
GENERATED: Question: A gentleman is carrying equipment for golf, what is he likely to have?
Choices: (A) club (B) assembly hall (C) meditation center (D) meeting (E) church
Answer: assembly hall
Explanation: a gentleman is carrying equipment for golf assembly hall. what is he likely to have assembly hall. assembly hall is a place where people can gather together to play golf.
PARSED: assembly hall
PRED IDX: 1 GOLD IDX: 0
GOLD ANSWER: club
--------------------------------------------------------------------------------


re-eval acc: 100%|██████████| 200/200 [02:21<00:00,  1.41it/s]


,model,lora_rank,val_acc@200,checkpoint_dir
0,baseline_gpt2,0,0.160,N/A
1,gpt2_lora,4,0.345,./lora_rank_ablation_outputs/gpt2_lora_r4
2,gpt2_lora,8,0.360,./lora_rank_ablation_outputs/gpt2_lora_r8
3,gpt2_lora,16,0.330,./lora_rank_ablation_outputs/gpt2_lora_r16



Saved fixed accuracy csv to: ./lora_rank_ablation_outputs/rank_ablation_results_fixed_acc.csv


**Summary.** The rank ablation compares ranks {4, 8, 16}. In the current notebook summary, LoRA improves over the GPT-2 baseline, and rank = 8 gives the best validation accuracy among the tested settings.

## Module 6. Chain-of-Thought (CoT) Prompting Experiment

This module repeats the training/evaluation pipeline under a CoT-style prompt design so it can be compared directly with the normal-prompt setting.

In [ ]:
# =========================
# Install dependencies
# =========================
!pip -q install torch transformers datasets peft tqdm bert-score

import random
import re
import string
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from datasets import load_dataset

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    default_data_collator,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model

# =========================
# Reproducibility
# =========================
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Config
# =========================
EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
MAX_LEN = 256
EXPL_KEY = "abstractive_explanation"
VAL_ACC_SAMPLES = 200
SAVE_DIR = "lora_gpt2_cose_cot_ckpt"

# =========================
# Helper functions
# =========================
def choices_to_list(choices):
    if isinstance(choices, (list, tuple)):
        return list(choices)
    if isinstance(choices, dict) and "text" in choices and isinstance(choices["text"], (list, tuple)):
        return list(choices["text"])
    return [str(choices)]

def choices_to_str(choices):
    opts = choices_to_list(choices)
    return " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(opts)])

# CoT prompt
def build_prompt(ex):
    return (
        f"Question: {ex['question']}\n"
        f"Choices: {choices_to_str(ex['choices'])}\n"
        f"Let's think step by step.\n"
        f"Answer:"
    )

def build_target(ex):
    return f" {ex['answer']}\nExplanation: {ex[EXPL_KEY]}"

def normalize_text(s):
    s = str(s).strip().lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def map_pred_to_answer_text(pred, choices):
    pred_norm = normalize_text(pred)
    opts = choices_to_list(choices)

    # case 1: single option letter
    m = re.match(r"^\(?([a-e])\)?$", pred_norm)
    if m:
        idx = ord(m.group(1)) - ord("a")
        if 0 <= idx < len(opts):
            return normalize_text(opts[idx])

    # case 2: option letter + text
    m = re.match(r"^\(?([a-e])\)?\s+(.*)$", pred_norm)
    if m:
        idx = ord(m.group(1)) - ord("a")
        if 0 <= idx < len(opts):
            return normalize_text(opts[idx])
        pred_norm = normalize_text(m.group(2))

    # case 3: direct text match
    for opt in opts:
        opt_norm = normalize_text(opt)
        if pred_norm == opt_norm or pred_norm in opt_norm or opt_norm in pred_norm:
            return opt_norm

    return pred_norm

def parse_answer(generated_text):
    if "Answer:" not in generated_text:
        return ""

    after = generated_text.split("Answer:", 1)[1].strip()

    if "Explanation:" in after:
        after = after.split("Explanation:", 1)[0].strip()

    first_line = after.split("\n", 1)[0].strip()
    return first_line

# =========================
# Load dataset
# =========================
train_ds = load_dataset("cos_e", "v1.11", split="train")
val_ds = load_dataset("cos_e", "v1.11", split="validation")

print("Train columns:", train_ds.column_names)
print("Train size:", len(train_ds), "| Val size:", len(val_ds))

# =========================
# Tokenizer
# =========================
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_example(ex):
    prompt = build_prompt(ex)
    target = build_target(ex)
    full_text = prompt + target

    tok = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LEN,
        add_special_tokens=False,
    )["input_ids"]

    prompt_len = min(len(prompt_ids), MAX_LEN)
    labels = tok["input_ids"].copy()

    # mask prompt part
    for i in range(prompt_len):
        labels[i] = -100

    # mask padding part
    labels = [lab if am == 1 else -100 for lab, am in zip(labels, tok["attention_mask"])]
    tok["labels"] = labels
    return tok

train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_example, remove_columns=val_ds.column_names)

train_loader = DataLoader(
    train_tok,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=default_data_collator,
)
val_loader = DataLoader(
    val_tok,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=default_data_collator,
)

# =========================
# Validation helpers
# =========================
def compute_val_loss_for(model_to_eval):
    was_training = model_to_eval.training
    model_to_eval.eval()

    total, n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            out = model_to_eval(
                input_ids=input_ids,
                attention_mask=attn,
                labels=labels
            )
            total += float(out.loss.detach().cpu().item())
            n += 1

    if was_training:
        model_to_eval.train()
    return total / max(n, 1)

def eval_val_accuracy_for(model_to_eval, n_samples=200):
    was_training = model_to_eval.training
    model_to_eval.eval()

    correct = 0
    m = min(n_samples, len(val_ds))

    with torch.no_grad():
        for i in range(m):
            ex = val_ds[i]
            prompt = build_prompt(ex)
            inputs = tokenizer(prompt, return_tensors="pt").to(device)

            gen_ids = model_to_eval.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            decoded = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

            pred = parse_answer(decoded)
            pred_norm = map_pred_to_answer_text(pred, ex["choices"])
            gold_norm = normalize_text(ex["answer"])

            if pred_norm == gold_norm:
                correct += 1

    if was_training:
        model_to_eval.train()
    return correct / m

# =========================
# Baseline GPT-2 with CoT prompt
# =========================
base_eval_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
base_val_loss = compute_val_loss_for(base_eval_model)
base_val_acc = eval_val_accuracy_for(base_eval_model, n_samples=VAL_ACC_SAMPLES)
print(f"[Baseline GPT-2 + CoT prompt] val_loss={base_val_loss:.4f} | val_acc@{VAL_ACC_SAMPLES}={base_val_acc:.3f}")

del base_eval_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =========================
# Model: GPT-2 + LoRA
# =========================
base_model = GPT2LMHeadModel.from_pretrained("gpt2")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config).to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# =========================
# Training loop
# =========================
best_val_loss = float("inf")
optimizer.zero_grad()

for ep in range(1, EPOCHS + 1):
    pbar = tqdm(train_loader, desc=f"CoT Epoch {ep}/{EPOCHS}")

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(
            input_ids=input_ids,
            attention_mask=attn,
            labels=labels
        )
        loss = out.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        pbar.set_postfix(train_loss=float(loss.detach().cpu().item()))

    val_loss = compute_val_loss_for(model)
    val_acc = eval_val_accuracy_for(model, n_samples=VAL_ACC_SAMPLES)

    print(f"[CoT Epoch {ep}] val_loss={val_loss:.4f} | val_acc@{VAL_ACC_SAMPLES}={val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"Best checkpoint updated -> {SAVE_DIR} (val_loss={best_val_loss:.4f})")

print("CoT training done. Best checkpoint:", SAVE_DIR)

Device: cuda
Train columns: ['id', 'question', 'choices', 'answer', 'abstractive_explanation', 'extractive_explanation']
Train size: 9741 | Val size: 1221


Map:   0%|          | 0/9741 [00:00<?, ? examples/s]

Map:   0%|          | 0/1221 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[Baseline GPT-2 + CoT prompt] val_loss=3.9378 | val_acc@200=0.060


...

[CoT Epoch 1] val_loss=2.1378 | val_acc@200=0.330
Best checkpoint updated -> lora_gpt2_cose_cot_ckpt (val_loss=2.1378)


CoT Epoch 2/3: 100%|██████████| 2436/2436 [02:15<00:00, 17.93it/s, train_loss=1.02]


[CoT Epoch 2] val_loss=2.0900 | val_acc@200=0.320
Best checkpoint updated -> lora_gpt2_cose_cot_ckpt (val_loss=2.0900)


CoT Epoch 3/3: 100%|██████████| 2436/2436 [02:15<00:00, 17.95it/s, train_loss=2.76]


[CoT Epoch 3] val_loss=2.0805 | val_acc@200=0.350
Best checkpoint updated -> lora_gpt2_cose_cot_ckpt (val_loss=2.0805)
CoT training done. Best checkpoint: lora_gpt2_cose_cot_ckpt


In [5]:
import os
import pandas as pd
from IPython.display import display

main_results = [
    {"Model": "GPT-2",        "Prompt": "Normal", "LoRA Rank": "-", "Setting": "Zero-shot",          "Val Accuracy@200": 0.16},
    {"Model": "GPT-2",        "Prompt": "CoT",    "LoRA Rank": "-", "Setting": "Zero-shot",          "Val Accuracy@200": 0.06},
    {"Model": "GPT-2 + LoRA", "Prompt": "Normal", "LoRA Rank": 8,   "Setting": "PEFT (LoRA)",       "Val Accuracy@200": 0.36},
    {"Model": "GPT-2 + LoRA", "Prompt": "CoT",    "LoRA Rank": 8,   "Setting": "PEFT (LoRA)",       "Val Accuracy@200": 0.35},
]

baseline_csv_candidates = [
    "./baseline_gpt2_cose_ckpt/baseline_results.csv",
    "baseline_gpt2_cose_ckpt/baseline_results.csv",
]

for baseline_csv in baseline_csv_candidates:
    if os.path.exists(baseline_csv):
        baseline_df = pd.read_csv(baseline_csv)
        if len(baseline_df) > 0:
            main_results.append({
                "Model": "GPT-2",
                "Prompt": "Normal",
                "LoRA Rank": "-",
                "Setting": "Full fine-tuning baseline",
                "Val Accuracy@200": float(baseline_df.loc[0, "val_accuracy_at_200"]),
            })
        break

main_df = pd.DataFrame(main_results)

print("=== Main Comparison Table ===")
display(main_df)

main_df.to_csv("main_comparison_results.csv", index=False)
print("Saved: main_comparison_results.csv")

=== Main Comparison Table ===


,Model,Prompt,LoRA Rank,Setting,Val Accuracy@200
0,GPT-2,Normal,-,Zero-shot,0.160
1,GPT-2,CoT,-,Zero-shot,0.060
2,GPT-2 + LoRA,Normal,8,PEFT (LoRA),0.360
3,GPT-2 + LoRA,CoT,8,PEFT (LoRA),0.350
4,GPT-2,Normal,-,Full fine-tuning baseline,0.305


Saved: main_comparison_results.csv


## Module 7. Error Analysis

This module collects misclassified validation examples and inspects the error cases, helping answer: why the model fails, what kinds of questions are harder, whether LoRA reduces some errors, and whether CoT changes the error pattern.

In [ ]:
import pandas as pd

def collect_errors(model, dataset, n_samples=200):

    errors = []

    model.eval()

    for i in range(min(n_samples, len(dataset))):

        ex = dataset[i]

        prompt = build_prompt(ex)

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
            )

        decoded = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

        pred = parse_answer(decoded)
        pred_norm = map_pred_to_answer_text(pred, ex["choices"])
        gold_norm = normalize_text(ex["answer"])

        if pred_norm != gold_norm:

            errors.append({
                "question": ex["question"],
                "choices": choices_to_list(ex["choices"]),
                "gold": ex["answer"],
                "prediction": pred,
            })

    return pd.DataFrame(errors)

In [ ]:
error_df = collect_errors(model, val_ds, 200)

error_df.head(20)

,question,choices,gold,prediction
0,"A beaver is know for building prowess, their s...","[british columbia, body of water, wooded area,...",wooded area,british columbia
1,"A farmer sees a weasel in the woods, where is ...","[chicken coop, beach, fairytale, great outdoor...",great outdoors,fairytale
2,"A gentleman is carrying equipment for golf, wh...","[club, assembly hall, meditation center, meeti...",club,assembly hall
3,"A human wants to submerge himself in water, wh...","[whirlpool bath, coffee cup, cup, soft drink, ...",whirlpool bath,soft drink
4,A man takes a seat at a museum outside of Barc...,"[in cinema, martorell, falling down, show, air...",martorell,show
5,A man wants air conditioning while we watches ...,"[car, house, offices, park, movie theatre]",house,movie theatre
6,"A potato is kept in the cellar, where is likel...","[farmer's market, grocery bag, pantry, bushel ...",bushel basket,pantry
7,A revolving door is convenient for two directi...,"[bank, library, department store, mall, new york]",bank,department store
8,A thoroughfare meandered through fields and wo...,"[move about, city, country, town, new york city]",country,new york city
9,"After climbing the mountains, the explored fou...","[west virginia, kentucky, desert, sea, rocky h...",rocky hills,west virginia


## Module 8. Backup and Output Folder Description

This module packages key output folders for download and records what each folder corresponds to in the experiment pipeline.

In [ ]:
import os
import zipfile
from google.colab import files

folders_to_backup = [
    "/content/lora_gpt2_cose_ckpt",
    "/content/baseline_gpt2_cose_ckpt",
    "/content/lora_gpt2_cose_cot_ckpt",
    "/content/lora_rank_ablation_outputs",
]

zip_name = "/content/lora_project_backup.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in folders_to_backup:
        if not os.path.exists(folder):
            print(f"Skip: {folder} not found")
            continue

        for root, dirs, files_in_dir in os.walk(folder):
            for file in files_in_dir:
                full_path = os.path.join(root, file)
                arcname = os.path.relpath(full_path, "/content")
                zipf.write(full_path, arcname)

print(f"Created zip: {zip_name}")

files.download(zip_name)

Created zip: /content/lora_project_backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Output folder description

| Folder | Experiment |
| --- | --- |
| `baseline_gpt2_cose_ckpt` | Full fine-tuning GPT-2 baseline + normal prompt |
| `lora_gpt2_cose_ckpt` | LoRA + normal prompt |
| `lora_gpt2_cose_cot_ckpt` | LoRA + CoT prompt |
| `lora_rank_ablation_outputs` | LoRA rank ablation |


In [6]:
# 打包所有需要的文件
!zip -r final_results.zip \
    lora_gpt2_cose_ckpt \
    baseline_gpt2_cose_ckpt \
    main_comparison_results.csv

from google.colab import files
files.download("final_results.zip")

  adding: lora_gpt2_cose_ckpt/ (stored 0%)
  adding: lora_gpt2_cose_ckpt/adapter_config.json (deflated 58%)
  adding: lora_gpt2_cose_ckpt/README.md (deflated 66%)
  adding: lora_gpt2_cose_ckpt/tokenizer_config.json (deflated 48%)
  adding: lora_gpt2_cose_ckpt/tokenizer.json (deflated 82%)
  adding: lora_gpt2_cose_ckpt/adapter_model.safetensors (deflated 7%)
  adding: baseline_gpt2_cose_ckpt/ (stored 0%)
  adding: baseline_gpt2_cose_ckpt/model.safetensors (deflated 7%)
  adding: baseline_gpt2_cose_ckpt/baseline_results.csv (deflated 7%)
  adding: baseline_gpt2_cose_ckpt/config.json (deflated 53%)
  adding: baseline_gpt2_cose_ckpt/baseline_training_history.csv (deflated 27%)
  adding: baseline_gpt2_cose_ckpt/tokenizer_config.json (deflated 48%)
  adding: baseline_gpt2_cose_ckpt/tokenizer.json (deflated 82%)
  adding: baseline_gpt2_cose_ckpt/generation_config.json (deflated 25%)
  adding: main_comparison_results.csv (deflated 35%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>